# InfoGAN — disentangling the latent code by maximizing mutual information

> Tutorial pair for [`infogan.py`](infogan.py).

## 1. Intuition
A vanilla GAN's noise vector $z$ is **entangled**: no single coordinate has an
interpretable meaning. InfoGAN splits the input into incompressible noise $z$ and
a small set of *structured* codes $c$, then rewards the generator for making $c$
**recoverable** from the output. The result: one code coordinate ends up
controlling one human-meaningful factor of variation — here, the angle around a
2-D ring — for free, with no labels.

## 2. Concept (the slide)
- **Generator** $G(z, c)$: noise $z$ plus a latent code $c$ produce a sample.
- **Mutual information** $I(c; G(z,c))$ measures how much the output tells us
  about $c$. Maximizing it forces $G$ to *use* $c$ rather than ignore it.
- $I$ is intractable (needs the true posterior $P(c\mid x)$), so we introduce an
  **auxiliary network $Q$** approximating it and maximize a **variational lower
  bound**. $Q$ shares its feature trunk with the discriminator $D$.
- For a **continuous** code, $Q$ outputs a Gaussian $(\mu, \log\sigma^2)$ and the
  bound reduces to a Gaussian log-likelihood of $c$.

## 3. Math derivation — the variational MI lower bound

Vanilla GAN keeps its adversarial term unchanged:
$$\min_{G}\max_{D}\; V(D,G)=\mathbb E_{x\sim p_{\text{data}}}[\log D(x)]
 +\mathbb E_{z,c}[\log(1-D(G(z,c)))].$$
InfoGAN **adds** a mutual-information regularizer, turning the objective into
$$\min_{G,Q}\max_{D}\; V(D,G)-\lambda\, I(c; G(z,c)).$$

**Why a bound is needed.** By definition
$$I(c; G(z,c)) = H(c) - H(c\mid G(z,c))
 = H(c) + \mathbb E_{x\sim G(z,c)}\big[\mathbb E_{c'\sim P(c\mid x)}[\log P(c'\mid x)]\big].$$
The posterior $P(c\mid x)$ is unknown. Introduce an auxiliary distribution
$Q(c\mid x)$; using the non-negativity of the KL divergence
$\mathrm{KL}\!\big(P(\cdot\mid x)\,\Vert\,Q(\cdot\mid x)\big)\ge 0$ gives the
**variational lower bound**
$$I(c; G(z,c)) \ge H(c) + \mathbb E_{c\sim P(c),\,x\sim G(z,c)}\big[\log Q(c\mid x)\big]
 \;=\; L_I(G, Q).$$
Since $H(c)$ is constant (we fix the code prior), maximizing $L_I$ means
maximizing $\mathbb E[\log Q(c\mid x)]$ — i.e. training $Q$ to **reconstruct the
code** from the generated sample, jointly with $G$.

**Continuous code.** With $Q(c\mid x)=\mathcal N\big(c;\,\mu(x),\,\sigma^2(x)\big)$,
$$-\log Q(c\mid x) = \tfrac12\sum_i\Big(\log\sigma_i^2 + \tfrac{(c_i-\mu_i)^2}{\sigma_i^2}\Big) + \text{const},$$
which is exactly the Gaussian NLL minimized in the code. **Why it differs from
vanilla GAN:** the extra term ties the latent to the output so the code becomes
disentangled and interpretable, whereas a plain GAN's latent is arbitrary.

## 4. Generator / key component

In [ ]:
# ===== actual implementation from infogan.py =====
from __future__ import annotations

import numpy as np

import torch

import torch.nn as nn

SEED = 0

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def make_ring(n: int = 2000, r: float = 2.0, seed: int = SEED) -> np.ndarray:
    rng = np.random.default_rng(seed)
    ang = rng.uniform(0, 2 * np.pi, n)
    pts = np.c_[r * np.cos(ang), r * np.sin(ang)]
    return (pts + 0.05 * rng.normal(size=(n, 2))).astype(np.float32)

class Generator(nn.Module):
    def __init__(self, noise_dim: int = 4, code_dim: int = 1, data_dim: int = 2,
                 hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim + code_dim, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, data_dim))

    def forward(self, z: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([z, c], dim=1))

class DiscriminatorQ(nn.Module):
    def __init__(self, data_dim: int = 2, code_dim: int = 1, hidden: int = 64):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(data_dim, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2, True))
        self.d_head = nn.Linear(hidden, 1)              # real/fake logit
        self.q_mu = nn.Linear(hidden, code_dim)         # posterior mean
        self.q_logvar = nn.Linear(hidden, code_dim)     # posterior log-variance

    def forward(self, x: torch.Tensor):
        h = self.trunk(x)
        return self.d_head(h), self.q_mu(h), self.q_logvar(h)

## 5. Trainer / losses

In [ ]:
# ===== actual implementation from infogan.py =====
class InfoGANTorch:
    def __init__(self, noise_dim: int = 4, code_dim: int = 1, data_dim: int = 2,
                 lr: float = 2e-4, lambda_mi: float = 0.5):
        torch.manual_seed(SEED)
        self.dev = get_device()
        self.noise_dim, self.code_dim, self.lambda_mi = noise_dim, code_dim, lambda_mi
        self.G = Generator(noise_dim, code_dim, data_dim).to(self.dev)
        self.DQ = DiscriminatorQ(data_dim, code_dim).to(self.dev)
        self.bce = nn.BCEWithLogitsLoss()
        # G and Q are optimized together (they jointly maximize the MI bound).
        self.optG = torch.optim.Adam(
            list(self.G.parameters()) + list(self.DQ.q_mu.parameters())
            + list(self.DQ.q_logvar.parameters()),
            lr=lr, betas=(0.5, 0.999))
        self.optD = torch.optim.Adam(self.DQ.parameters(), lr=lr, betas=(0.5, 0.999))

    def _sample_code(self, batch: int) -> torch.Tensor:
        # continuous code ~ U(-1, 1)
        return torch.rand(batch, self.code_dim, device=self.dev) * 2 - 1

    @staticmethod
    def _gaussian_nll(c: torch.Tensor, mu: torch.Tensor, logvar: torch.Tensor):
        """Negative log-likelihood of c under N(mu, exp(logvar)) (the -Q term)."""
        return 0.5 * (logvar + (c - mu) ** 2 / logvar.exp()).sum(1).mean()

    def fit(self, real: np.ndarray, steps: int = 1500, batch: int = 128):
        real = torch.as_tensor(real, dtype=torch.float32, device=self.dev)
        ones = torch.ones(batch, 1, device=self.dev)
        zeros = torch.zeros(batch, 1, device=self.dev)
        self.d_hist, self.g_hist, self.mi_hist = [], [], []
        for _ in range(steps):
            idx = torch.randint(0, len(real), (batch,), device=self.dev)
            x = real[idx]
            # --- D step ---
            z = torch.randn(batch, self.noise_dim, device=self.dev)
            c = self._sample_code(batch)
            fake = self.G(z, c).detach()
            d_real, _, _ = self.DQ(x)
            d_fake, _, _ = self.DQ(fake)
            lossD = self.bce(d_real, ones) + self.bce(d_fake, zeros)
            self.optD.zero_grad(); lossD.backward(); self.optD.step()
            # --- G + Q step: fool D AND let Q recover the code (MI bound) ---
            z = torch.randn(batch, self.noise_dim, device=self.dev)
            c = self._sample_code(batch)
            gen = self.G(z, c)
            d_g, q_mu, q_logvar = self.DQ(gen)
            lossG = self.bce(d_g, ones)
            mi = self._gaussian_nll(c, q_mu, q_logvar)  # minimizing NLL maximizes MI bound
            (lossG + self.lambda_mi * mi).backward()
            self.optG.step(); self.optG.zero_grad()
            self.d_hist.append(lossD.item()); self.g_hist.append(lossG.item())
            self.mi_hist.append(mi.item())
        return self

    @torch.no_grad()
    def generate(self, n: int, code: float | None = None) -> np.ndarray:
        z = torch.randn(n, self.noise_dim, device=self.dev)
        if code is None:
            c = self._sample_code(n)
        else:
            c = torch.full((n, self.code_dim), float(code), device=self.dev)
        return self.G(z, c).cpu().numpy()

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    real = make_ring(2000)

    gan = InfoGANTorch().fit(real, steps=1500, batch=128)
    print(f"D loss {np.mean(gan.d_hist[:100]):.3f} -> {np.mean(gan.d_hist[-100:]):.3f}")
    print(f"Q NLL  {np.mean(gan.mi_hist[:100]):.3f} -> {np.mean(gan.mi_hist[-100:]):.3f}"
          f"  (falling => MI bound tightening, code is recoverable)")

    # Disentanglement check: sweeping the code should sweep the ring angle.
    angles = []
    for cval in np.linspace(-1, 1, 9):
        s = gan.generate(200, code=cval)
        angles.append(np.arctan2(s[:, 1].mean(), s[:, 0].mean()))
    angles = np.unwrap(angles)
    span = angles.max() - angles.min()
    print(f"angle span as code goes -1->1: {np.degrees(span):.0f} deg "
          f"(large => the code controls position on the ring)")

## 6. Train

In [ ]:
demo()

## 7. Visualization

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import infogan as M

real = M.make_ring(2000)
gan = M.InfoGANTorch().fit(real, steps=1500, batch=128)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
# Left: sweeping the code should sweep position on the ring (disentanglement).
cmap = plt.get_cmap("viridis")
codes = np.linspace(-1, 1, 9)
ax[0].scatter(real[:, 0], real[:, 1], s=6, alpha=.12, color="gray", label="real")
for i, cval in enumerate(codes):
    s = gan.generate(120, code=cval)
    ax[0].scatter(s[:, 0], s[:, 1], s=10, alpha=.7, color=cmap(i / (len(codes) - 1)))
ax[0].set_title("Code -1 -> +1 sweeps the ring angle"); ax[0].set_aspect("equal")
ax[0].legend()
# Right: Q's NLL falling means the MI bound is tightening (code recoverable).
ax[1].plot(gan.mi_hist, alpha=.7)
ax[1].set_xlabel("step"); ax[1].set_ylabel("Q NLL")
ax[1].set_title("Variational MI bound tightening (lower = better)")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- InfoGAN keeps the adversarial game but **adds** a variational MI term so a code
  coordinate controls a real factor of variation — disentanglement without labels.
- $Q$ sharing $D$'s trunk is nearly free; only the small posterior heads are extra.
- Pitfalls: $\lambda$ too large overpowers the adversarial loss (blurry samples);
  too small and $G$ ignores the code. Continuous codes need a sensible prior
  ($U(-1,1)$ here); discrete codes use a categorical $Q$ and cross-entropy instead.